In [1]:
import os
from argparse import Namespace
from typing import NamedTuple

from huggingface_hub import snapshot_download
from PIL.Image import Image
from transformers import AutoProcessor, AutoTokenizer

from vllm import LLM, EngineArgs, SamplingParams
from vllm.lora.request import LoRARequest
from vllm.multimodal.utils import fetch_image
from vllm.platforms import current_platform
from vllm.utils.argparse_utils import FlexibleArgumentParser

import time
from PIL import Image

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import json
import datetime

In [3]:
%run DS_click_1player_perspective.ipynb

Loading logs from logs...
2503 files found.
DONE. Loaded 2503 completed game logs.


In [4]:
print(click_1persp_DS_100['1834_6_1_A'][2])

A


In [6]:
TESTED_MODEL = "qwen2_5_vl"

In [7]:
# QUESTION = "How many images do you see? Describe each of them"

In [8]:
QUESTION = """
You are a helpful language and vision assistant. You see a chat between two people, A and B.
They are playing a game in which they are seeing a set of 6 images from the photobook album.
Their task is to find out which photos are common for both of them, and which are different.
They are chatting between each other.
6 images you received are images that player A sees. 
At this point of conversation player A marks one of the images as common. Which one is it?
"""

In [9]:
class ModelRequestData(NamedTuple):
    engine_args: EngineArgs
    prompt: str
    image_data: list[Image]
    stop_token_ids: list[int] | None = None
    chat_template: str | None = None
    lora_requests: list[LoRARequest] | None = None
    sampling_params: SamplingParams | None = None


In [10]:
def load_qwen2_5_vl(question: str, image_urls: list[str]) -> ModelRequestData:
    try:
        from qwen_vl_utils import smart_resize
    except ModuleNotFoundError:
        print(
            "WARNING: `qwen-vl-utils` not installed, input images will not "
            "be automatically resized. You can enable this functionality by "
            "`pip install qwen-vl-utils`."
        )
        smart_resize = None

    model_name = "Qwen/Qwen2.5-VL-3B-Instruct"

    engine_args = EngineArgs(
        model=model_name,
        max_model_len=32768 if smart_resize is None else 4096,
        max_num_seqs=5,
        limit_mm_per_prompt={"image": len(image_urls)},
    )


    # placeholders = [{"type": "image", "image": url} for url in image_urls]

    placeholders = []
    for i, url in enumerate(image_urls, 1):
        placeholders.append({"type": "text", "text": f"Image {i}:"})
        placeholders.append({"type": "image", "image": url})
    
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {
            "role": "user",
            "content": [
                *placeholders,
                {"type": "text", "text": question},
            ],
        },
    ]

    processor = AutoProcessor.from_pretrained(model_name)

    prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    if smart_resize is None:
        image_data = [fetch_image(url) for url in image_urls]
    else:
        def post_process_image(image: Image) -> Image:
            width, height = image.size
            resized_height, resized_width = smart_resize(
                height, width, factor=28, max_pixels=1024 * 28 * 28
            )
            return image.resize((resized_width, resized_height))

        image_data = [post_process_image(fetch_image(url)) for url in image_urls]

    return (ModelRequestData(
        engine_args=engine_args,
        prompt=prompt,
        image_data=image_data),
        messages)

In [11]:
model_example_map = {
     "qwen2_5_vl": load_qwen2_5_vl,
}

In [12]:
def run_generate(
    model,
    question: str,
    image_urls: list[str],
    seed: int,
    tensor_parallel_size: int | None,
):
    (req_data, message_documentation) = model_example_map[model](question, image_urls)

    engine_args = req_data.engine_args
    engine_args.seed = seed
    if tensor_parallel_size is not None:
        engine_args.tensor_parallel_size = tensor_parallel_size
    if current_platform.is_rocm():
        os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
    llm = LLM.from_engine_args(engine_args)

    sampling_params = SamplingParams(
        temperature=0.0, max_tokens=1500, stop_token_ids=req_data.stop_token_ids
    )

    outputs = llm.generate(
        {
            "prompt": req_data.prompt,
            "multi_modal_data": {"image": req_data.image_data},
        },
        sampling_params=sampling_params,
        lora_request=req_data.lora_requests,
    )

    list_of_outputs = []
    
    print("-" * 50)
    for o in outputs:
        generated_text = o.outputs[0].text
        print(generated_text)
        print("-" * 50)
        list_of_outputs.append(generated_text)
    list_of_outputs.append(message_documentation)    
    return list_of_outputs


In [13]:
def parse_args(IMAGE_URLS):
    parser = FlexibleArgumentParser(
        description="Demo on using vLLM for offline inference with "
        "vision language models that support multi-image input for text "
        "generation"
    )
    parser.add_argument(
        "--model-type",
        "-m",
        type=str,
        default=TESTED_MODEL,
        choices=model_example_map.keys(),
        help='Huggingface "model_type".',
    )
    parser.add_argument(
        "--method",
        type=str,
        default="generate",
        choices=["generate", "chat"],
        help="The method to run in `vllm.LLM`.",
    )
    parser.add_argument(
        "--seed",
        type=int,
        default=0,
        help="Set the seed when initializing `vllm.LLM`.",
    )
    parser.add_argument(
        "--num-images",
        "-n",
        type=int,
        choices=list(range(1, len(IMAGE_URLS) + 1)),  # the max number of images
        default=len(IMAGE_URLS),
        # default=2,
        help="Number of images to use for the demo.",
    )
    parser.add_argument(
        "--tensor-parallel-size",
        "-tp",
        type=int,
        default=None,
        help="Tensor parallel size to override the model's default setting. ",
    )
    parser.add_argument('-f', required=False,)
    
    # DELETE PRINT
    # print(parser.parse_args())
    
    return parser.parse_args()


In [14]:
def main(args: Namespace, IMAGE_URLS, task):
    model = args.model_type
    method = args.method
    seed = args.seed
    tensor_parallel_size = args.tensor_parallel_size

    if tensor_parallel_size is not None and tensor_parallel_size < 1:
        raise ValueError(
            f"tensor_parallel_size must be a positive integer, "
            f"got {tensor_parallel_size}"
        )

    image_urls = IMAGE_URLS[: args.num_images]

    if method == "generate":
        return run_generate(model, task, image_urls, seed, tensor_parallel_size)
    elif method == "chat":
        run_chat(model, task, image_urls, seed, tensor_parallel_size)
    else:
        raise ValueError(f"Invalid method: {method}")


In [15]:
opposite = {"A":"B", "B":"A"}

In [17]:
def run_baseline(data_set, model):
    counter = 20
    results = {}
    for key in data_set:
        if counter > 0:
            print(counter)
            
            tested_player = data_set[key][2]
            marked_label = data_set[key][4]

            imgs_no_url = data_set[key][5][tested_player]
            imgs_url = ['file://images/' + no_url for no_url in imgs_no_url ]
            

            description = f"""
You are **Player {tested_player}** in a game with **Player {opposite[tested_player]}**.

### Game setup
You see six images. The six images provided to you are the six images visible to you.

Some images are visible to both you and the other player, while others are visible only to one of you.

The goal of the game is to identify which images are **common** and which are **different**:
- **Common:** The image is visible to both you and the other player.
- **Different:** The image is visible only to you and is not visible to the other player.

### Your task
You are given your six images and the chat history between you and the other player.

You do not participate in the conversation. Instead, you must use the chat history to understand the information you have gathered so far.

During the conversation, you and the other player exchange information about your images to determine which images are common and which are different. The chat history may also contain instances where you have already labeled some images as common or different. Take these existing labels into account when determining which image to select.

You have just gathered enough information to label one of your images as **{marked_label}**.

### Instructions
1. Review your six images.
2. Analyze the chat history using the information exchanged between you and the other player.
3. Take into account any images you have already labeled as common or different. Do not select an image that you have already labeled.
4. Determine which remaining image you would label as **{marked_label}**, based on the information available to you at this point in the conversation.

**Which image are you about to label?**

Return only the image identifier or number. Do not provide any explanation.
"""
            
            start_time = time.time()

            args = parse_args(imgs_url)
            outputs_list = main(args, imgs_url, description)

            how_long = "--- %s seconds ---" % (time.time() - start_time)
            print(how_long)
            

            results[key] = (outputs_list, data_set[key], how_long)

            counter -= 1
    now = str(datetime.datetime.now())
    filename = "output _logs/baseline/baseline_" + model + "_" + now + ".json"        
    with open(filename, "w") as f:
            json.dump(results, f, indent=4)

testing = run_baseline(click_1persp_DS_100, "qwen2_5_vl")

20
INFO 10-23 15:41:00 [api_utils.py:286] non-default args: {'max_model_len': 4096, 'max_num_seqs': 5, 'limit_mm_per_prompt': {'image': 6}, 'model': 'Qwen/Qwen2.5-VL-3B-Instruct'}
INFO 10-23 15:41:00 [model.py:684] Resolved architecture: Qwen2_5_VLForConditionalGeneration
INFO 10-23 15:41:00 [model.py:2021] Using max model len 4096
INFO 10-23 15:41:00 [kernel.py:369] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=2502093) INFO 10-23 15:41:02 [core.py:123] Initializing a V1 LLM engine (v0.29.0) with config: model='Qwen/Qwen2.5-VL-3B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-VL-3B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=main, tokenizer_revision=main, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


(EngineCore pid=2502093) INFO 10-23 15:41:12 [weight_utils.py:797] Prefetching checkpoint files: 10% (1/2)
(EngineCore pid=2502093) INFO 10-23 15:41:12 [weight_utils.py:797] Prefetching checkpoint files: 20% (2/2)
(EngineCore pid=2502093) INFO 10-23 15:41:12 [weight_utils.py:820] Prefetching checkpoint files into page cache finished in 0.26s


Loading safetensors checkpoint shards:  50% Completed | 1/2 [00:00<00:00,  1.98it/s]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:00<00:00,  2.22it/s]
Loading safetensors checkpoint shards: 100% Completed | 2/2 [00:00<00:00,  2.18it/s]
(EngineCore pid=2502093) 


(EngineCore pid=2502093) INFO 10-23 15:41:12 [default_loader.py:430] Loading weights took 0.95 seconds
(EngineCore pid=2502093) INFO 10-23 15:41:13 [model_runner.py:404] Model loading took 7.16 GiB memory and 5.453301 seconds
(EngineCore pid=2502093) INFO 10-23 15:41:13 [topk_topp_sampler.py:62] Using FlashInfer for top-p & top-k sampling.
(EngineCore pid=2502093) INFO 10-23 15:41:13 [utils.py:306] Using LBNHC KV cache layout.
(EngineCore pid=2502093) INFO 10-23 15:41:14 [encoder_runner.py:131] Encoder cache will be initialized with a budget of 16384 tokens, and profiled with 1 video items of the maximum feature size.
(EngineCore pid=2502093) INFO 10-23 15:41:16 [caching.py:343] reconstructed serializable fn from standalone compile artifacts. num_artifacts=4 num_submods=37
(EngineCore pid=2502093) INFO 10-23 15:41:16 [decorators.py:313] Directly load AOT compilation from path /home/slorys/.cache/vllm/torch_compile_cache/torch_aot_compile/13eb649725b70c4da5a60d8e19f570c057b9dd8ced5bc9d3

...

(EngineCore pid=2579181) 2026-10-23 15:53:29,648 - INFO - autotuner.py:995 - flashinfer.jit: [Autotuner]: Autotuning process ends
(EngineCore pid=2579181) 2026-10-23 15:53:29,654 - INFO - autotuner.py:2652 - flashinfer.jit: [Autotuner]: Saved 0 configs to /home/slorys/.cache/vllm/flashinfer_autotune_cache/0.6.18/90a/2666c1e6dac7e4ba8f0f89e8513600c269ab7d3b133254e9233dde09cff61860/autotune_configs.json (0 new, 0 from previous config)
Capturing CUDA graphs (FULL): 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 28.04it/s]


(EngineCore pid=2579181) INFO 10-23 15:53:30 [model_runner.py:960] Graph capturing finished in 1 secs, took 0.04 GiB
(EngineCore pid=2579181) INFO 10-23 15:53:30 [gpu_worker.py:797] CUDA graph pool memory: 0.04 GiB (actual), 0.06 GiB (estimated), difference: 0.02 GiB (52.6%).
(EngineCore pid=2579181) INFO 10-23 15:53:30 [gpu_worker.py:860] Free memory on device (94.44/95.0 GiB) on startup. Desired GPU memory utilization is (0.92, 87.4 GiB). Actual usage is 7.89 GiB for consumed memory (weights + non-torch), 2.32 GiB for peak activation, and 0.04 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=82690409882` (77.01 GiB) to fit into requested memory, or `--kv-cache-memory=90253028352` (84.05 GiB) to fully utilize gpu memory. Current kv cache memory in use is 77.2 GiB.
(EngineCore pid=2579181) INFO 10-23 15:53:31 [jit_monitor.py:85] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
(EngineCore pid=2579181) I

Processed prompts: 100%|█████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  5.58it/s, est. speed input: 13583.00 toks/s, output: 11.19 toks/s]

--------------------------------------------------
6
--------------------------------------------------
INFO 10-23 15:53:42 [utils.py:620] [shutdown] Process manager: send sigterm to process EngineCore
(EngineCore pid=2579181) INFO 10-23 15:53:42 [core.py:1350] [shutdown] EngineCore: trigger received signal=SIGTERM
(EngineCore pid=2579181) INFO 10-23 15:53:42 [core.py:1501] [shutdown] EngineCore: start mode=abort timeout=0s
(EngineCore pid=2579181) INFO 10-23 15:53:42 [core.py:1532] [shutdown] EngineCore: request processing complete; starting resource teardown
(EngineCore pid=2579181) INFO 10-23 15:53:42 [core.py:1363] [shutdown] EngineCore: exiting busy loop


--- 38.95283508300781 seconds ---
